# Session 6 — Comparing Two Groups Without Fooling Yourself

**Goal of this session:** catch the single most common real mistake in this literature — comparing a graph metric between two groups whose networks differ only in *density*, and mistaking that for a difference in *organisation*.

*Network Neuroscience in Python, session 6 of 10.*

## Why this matters

Nearly every graph metric you've met in this series — clustering coefficient especially — mechanically depends on how many edges a network has, independent of how those edges are arranged. If one group's connectivity matrices happen to be sparser than another's for a reason that has nothing to do with the biology you care about (different scanner, more head motion, a stricter preprocessing pipeline, simply thresholding at the same *correlation* cutoff when one group has noisier data), a metric like clustering will differ between groups even when the underlying wiring principles are identical. Papers get this wrong. Reviewers increasingly catch it. This session builds the trap ourselves so you can recognise it on sight, then builds the fix.

## Two groups, built to have zero real difference

We generate 20 "subjects" per group from `generate_toy_network()`, varying only the random seed — every subject in both groups is drawn from the *exact same* generative process (same `p_within`, same `p_between`, same module structure). By construction, there is no true organisational difference between group A and group B' anywhere in this cell.

Group B' is then built by randomly deleting a fixed fraction of each group-A subject's edges. That's it — no change to which nodes prefer which neighbours, no change to modularity, nothing. Just fewer edges, the way a noisier scan or a stricter preprocessing step might leave you with fewer edges by accident.

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from scipy import stats


def generate_toy_network(n_per_module=5, n_modules=2, p_within=0.7, p_between=0.05,
                          add_connector=True, connector_frac=0.6, seed=0):
    """Same function as session 1. Not a brain."""
    rng = np.random.default_rng(seed)
    G = nx.Graph()
    node_id = 0
    modules = []
    for m in range(n_modules):
        members = []
        for _ in range(n_per_module):
            G.add_node(node_id, module=m)
            members.append(node_id)
            node_id += 1
        modules.append(members)
    for m in range(n_modules):
        members = modules[m]
        for i in range(len(members)):
            for j in range(i + 1, len(members)):
                if rng.random() < p_within:
                    G.add_edge(members[i], members[j])
    for m1 in range(n_modules):
        for m2 in range(m1 + 1, n_modules):
            for u in modules[m1]:
                for v in modules[m2]:
                    if rng.random() < p_between:
                        G.add_edge(u, v)
    if add_connector:
        connector = node_id
        G.add_node(connector, module="connector")
        n_link = max(1, round(connector_frac * n_per_module))
        for members in modules:
            chosen = rng.choice(members, size=min(n_link, len(members)), replace=False)
            for other in chosen:
                G.add_edge(connector, int(other))
    return G


def thin_graph(G, keep_frac, seed):
    """G with a random keep_frac fraction of its edges kept, same nodes."""
    rng = np.random.default_rng(seed)
    edges = list(G.edges())
    n_keep = max(1, round(keep_frac * len(edges)))
    keep_idx = rng.choice(len(edges), size=n_keep, replace=False)
    Gt = nx.Graph()
    Gt.add_nodes_from(G.nodes(data=True))
    for i in keep_idx:
        Gt.add_edge(*edges[i])
    return Gt


N_SUBJ = 20
group_A = [generate_toy_network(seed=i) for i in range(N_SUBJ)]
group_Bp = [thin_graph(group_A[i], keep_frac=0.55, seed=1000 + i) for i in range(N_SUBJ)]

density_A = np.array([nx.density(g) for g in group_A])
density_B = np.array([nx.density(g) for g in group_Bp])
print(f"group A  density: {density_A.mean():.3f} +/- {density_A.std():.3f}")
print(f"group B' density: {density_B.mean():.3f} +/- {density_B.std():.3f}")

## The naive comparison

The obvious thing to do: compute clustering coefficient for every subject in both groups, at whatever density each subject's network happens to have, and run a t-test.

In [ ]:
clustering_A = np.array([nx.average_clustering(g) for g in group_A])
clustering_B = np.array([nx.average_clustering(g) for g in group_Bp])

t_naive, p_naive = stats.ttest_ind(clustering_A, clustering_B)
pooled_sd = np.sqrt((clustering_A.std(ddof=1) ** 2 + clustering_B.std(ddof=1) ** 2) / 2)
d_naive = (clustering_A.mean() - clustering_B.mean()) / pooled_sd

print(f"clustering, group A:  {clustering_A.mean():.3f} +/- {clustering_A.std():.3f}")
print(f"clustering, group B': {clustering_B.mean():.3f} +/- {clustering_B.std():.3f}")
print(f"\nnaive t-test:  t={t_naive:.2f}   p={p_naive:.2e}   Cohen's d={d_naive:.2f}")

That is a huge effect size and a minuscule p-value. If you stopped here, this looks like a slam-dunk finding: "group B' shows significantly reduced clustering". It would also be **completely wrong** — we built these two groups from the identical generative process. What the test actually detected is that group B' has about 45% fewer edges, and clustering coefficient mechanically falls as density falls. There is no organisational difference here to find, because we didn't put one in.

## Seeing the trap visually

Plot clustering against density for every single subject in both groups. They fall on essentially one shared curve — density explains almost all of the spread. The naive comparison above picked one point on that shared curve for group A and a different point on the same curve for group B', and mistook the curve's own slope for a group difference.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(density_A, clustering_A, s=70, color="#2b6cb0", edgecolor="white",
           linewidth=0.8, label="group A subjects", zorder=3)
ax.scatter(density_B, clustering_B, s=70, color="#c53030", edgecolor="white",
           linewidth=0.8, label="group B' subjects", zorder=3)
ax.scatter([density_A.mean()], [clustering_A.mean()], s=320, marker="X",
           color="#2b6cb0", edgecolor="black", linewidth=1.5, zorder=4)
ax.scatter([density_B.mean()], [clustering_B.mean()], s=320, marker="X",
           color="#c53030", edgecolor="black", linewidth=1.5, zorder=4,
           label="group means (the naive comparison)")
ax.set_xlabel("network density", fontsize=12)
ax.set_ylabel("clustering coefficient", fontsize=12)
ax.set_title("Both groups sit on the same density-clustering curve", fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## The fix: compare across a shared density range

We already have the tool for this — the same edge-thinning function we just used to build group B'. For every subject in *both* groups, we sweep them down to a common set of target densities, low enough that every single subject (in either group) can actually reach every point on the sweep. You cannot threshold a network up to a density it never had; you can only threshold down, so the shared range is capped by whichever subject is sparsest in either group.

At each shared density, we average several random thinnings per subject (to smooth out the noise thinning itself introduces), then compare groups at that density with a fresh t-test.

In [ ]:
def clustering_at_density(G, target_density, n_repeats=8, seed=0):
    """Average clustering coefficient after thinning G down to target_density."""
    n_nodes = G.number_of_nodes()
    max_edges = n_nodes * (n_nodes - 1) / 2
    keep_frac = min(1.0, (target_density * max_edges) / G.number_of_edges())
    vals = [nx.average_clustering(thin_graph(G, keep_frac, seed=seed * 100 + r))
            for r in range(n_repeats)]
    return np.mean(vals)


target_densities = np.linspace(0.08, 0.20, 7)  # safely below every subject's native density

curve_A = np.array([[clustering_at_density(g, d, seed=i) for d in target_densities]
                     for i, g in enumerate(group_A)])
curve_B = np.array([[clustering_at_density(g, d, seed=2000 + i) for d in target_densities]
                     for i, g in enumerate(group_Bp)])

print("density   A mean (se)      B' mean (se)      t       p")
for j, d in enumerate(target_densities):
    mA, seA = curve_A[:, j].mean(), curve_A[:, j].std() / np.sqrt(N_SUBJ)
    mB, seB = curve_B[:, j].mean(), curve_B[:, j].std() / np.sqrt(N_SUBJ)
    t, p = stats.ttest_ind(curve_A[:, j], curve_B[:, j])
    print(f"{d:.2f}      {mA:.3f} ({seA:.3f})     {mB:.3f} ({seB:.3f})     {t:+.2f}   {p:.3f}")

## The false difference disappears

Overlay the two group-average curves across the shared density range, with error bars (standard error across subjects). We also mark exactly where the naive single-threshold comparison landed, so you can see precisely why it went wrong: it compared group A at *its* native density to group B' at a *different, lower* native density, off two different points on curves that, at any matched density, agree with each other within noise.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
mean_A, se_A = curve_A.mean(axis=0), curve_A.std(axis=0) / np.sqrt(N_SUBJ)
mean_B, se_B = curve_B.mean(axis=0), curve_B.std(axis=0) / np.sqrt(N_SUBJ)

ax.errorbar(target_densities, mean_A, yerr=se_A, marker="o", color="#2b6cb0",
            linewidth=2, capsize=4, label="group A (density-matched sweep)")
ax.errorbar(target_densities, mean_B, yerr=se_B, marker="o", color="#c53030",
            linewidth=2, capsize=4, label="group B' (density-matched sweep)")

ax.scatter([density_A.mean()], [clustering_A.mean()], s=260, marker="X", color="#2b6cb0",
           edgecolor="black", linewidth=1.5, zorder=5)
ax.scatter([density_B.mean()], [clustering_B.mean()], s=260, marker="X", color="#c53030",
           edgecolor="black", linewidth=1.5, zorder=5)
ax.annotate("the naive comparison\n(two different densities)",
            xy=(density_A.mean(), clustering_A.mean()), xytext=(-40, 30),
            textcoords="offset points", fontsize=10,
            arrowprops=dict(arrowstyle="->", color="gray"))

ax.set_xlabel("network density", fontsize=12)
ax.set_ylabel("clustering coefficient", fontsize=12)
ax.set_title("Once density is matched, the group difference vanishes", fontsize=13)
ax.legend(fontsize=10, loc="upper left")
plt.tight_layout()
plt.show()

t_mid, p_mid = stats.ttest_ind(curve_A[:, 3], curve_B[:, 3])
print(f"at a matched density of {target_densities[3]:.2f}: t={t_mid:+.2f}, p={p_mid:.3f}")
print(f"(compare to the naive result: t={t_naive:.2f}, p={p_naive:.2e})")

## The lesson

Every p-value along the density-matched sweep comes back non-significant — because there was never a real difference to find. The only thing the naive comparison detected was that we thinned one group's edges more than the other's, which we did on purpose to build this trap. In a real study, that same pattern can appear by accident: motion-heavy participants losing more edges after scrubbing, one site's scanner producing systematically sparser connectomes than another's, or simply picking one correlation threshold when the two groups' correlation distributions aren't matched.

The general fix is always one of: report metrics across a range of densities rather than one threshold, density-match your networks before comparing (proportional thresholding to the same edge count), or use a metric that is provably density-invariant (rare, and worth checking rather than assuming). What you should never do is compare one group's metric at its native density to another's at a different native density and call the gap organisational.

**Next session:** we leave the toy network behind for good. First real data — diffusion MRI, turned into an actual structural connectome.